In [ ]:
import json
from pathlib import Path

def merge_citation_data():
    """Merge citation data into the paper data"""
    
    # File paths
    citation_file = "citation_analysis_results.json"
    papers_file = "Data/DBLP_AI/ai_papers_20250831_215837.jsonl"
    output_file = "Data/DBLP_AI/ai_papers_with_citations.jsonl"
    
    print("Starting to merge citation data...")
    
    # Read the citation data
    print("Reading citation data...")
    with open(citation_file, 'r', encoding='utf-8') as f:
        citation_data = json.load(f)
    
    print(f"Loaded {len(citation_data)} citation records")
    
    # Statistics counters
    total_papers = 0
    matched_papers = 0
    unmatched_papers = 0
    
    # Process the paper data and merge in the citation information
    print("Processing paper data and merging citation information...")
    with open(papers_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            if line.strip():
                paper = json.loads(line.strip())
                total_papers += 1
                
                paper_id = paper.get('id')
                
                # Check whether there is corresponding citation data
                if paper_id and paper_id in citation_data:
                    # Merge the citation data, adding per-year citation counts
                    citation_info = citation_data[paper_id]
                    
                    # Add citation data fields
                    paper['citations_year_1'] = citation_info.get('citations_year_1')
                    paper['citations_year_2'] = citation_info.get('citations_year_2')
                    paper['citations_year_3'] = citation_info.get('citations_year_3')
                    paper['citations_year_4'] = citation_info.get('citations_year_4')
                    paper['citations_year_5'] = citation_info.get('citations_year_5')
                    
                    # Verify that publication_year is consistent
                    if 'publication_year' in citation_info:
                        citation_year = citation_info['publication_year']
                        paper_year = paper.get('year')
                        if paper_year != citation_year:
                            print(f"Warning: paper {paper_id} has inconsistent year - in paper: {paper_year}, in citation data: {citation_year}")
                    
                    matched_papers += 1
                else:
                    # Unmatched paper, add default citation data
                    paper['citations_year_1'] = 0
                    paper['citations_year_2'] = 0
                    paper['citations_year_3'] = 0
                    paper['citations_year_4'] = 0
                    paper['citations_year_5'] = 0
                    unmatched_papers += 1
                
                # Write the result
                outfile.write(json.dumps(paper, ensure_ascii=False) + '\n')
                
                # Progress display
                if total_papers % 10000 == 0:
                    print(f"Processed {total_papers} papers, matched: {matched_papers}, unmatched: {unmatched_papers}")
    
    # Output the statistics
    print("\n=== Merge complete ===")
    print(f"Total papers: {total_papers}")
    print(f"Matched: {matched_papers} ({matched_papers/total_papers*100:.2f}%)")
    print(f"Unmatched: {unmatched_papers} ({unmatched_papers/total_papers*100:.2f}%)")
    print(f"Output file: {output_file}")
    
    # Verify the output file and show an example
    print("\nVerifying output file...")
    with open(output_file, 'r', encoding='utf-8') as f:
        # Read the first successfully matched example
        for line in f:
            paper = json.loads(line.strip())
            if paper.get('citations_year_1', 0) > 0 or paper.get('citations_year_2', 0) > 0:
                print(f"\nExample of a successfully matched paper (ID: {paper.get('id')}):")
                print(f"  Title: {paper.get('title', '')[:50]}...")
                print(f"  Year: {paper.get('year')}")
                print(f"  Year-1 citations: {paper.get('citations_year_1', 0)}")
                print(f"  Year-2 citations: {paper.get('citations_year_2', 0)}")
                print(f"  Year-3 citations: {paper.get('citations_year_3', 0)}")
                print(f"  Year-4 citations: {paper.get('citations_year_4', 0)}")
                print(f"  Year-5 citations: {paper.get('citations_year_5', 0)}")
                break
    
    print("\nNew field descriptions:")
    print("  - citations_year_1: number of citations in the 1st year after publication")
    print("  - citations_year_2: number of citations in the 2nd year after publication")
    print("  - citations_year_3: number of citations in the 3rd year after publication")
    print("  - citations_year_4: number of citations in the 4th year after publication")
    print("  - citations_year_5: number of citations in the 5th year after publication")
    
    return matched_papers, unmatched_papers, total_papers

# Run the merge
matched, unmatched, total = merge_citation_data()
